In [1]:
!pip install trackio peft bitsandbytes trl datasets

  Using cached orjson-3.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (42 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 2.5 MB/s eta 0:00:00a 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 4.2 MB/s eta 0:00:00
Using cached orjson-3.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (130 kB)


#### Import Required Modules

In [3]:
import trackio # Experiment tracking dashboard
import re # Regex patterns for reward functions
import torch

from trl import GRPOConfig, GRPOTrainer

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import LoraConfig, get_peft_model, TaskType

from datasets import load_dataset

# logging configuration
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('gradio_client').setLevel(logging.WARNING)


#### Model and Configuration

In [40]:
model_name = "Qwen/Qwen2.5-3B-Instruct"
max_seq_length = 1024


# Configure 4-bit quantization for ~75% memory reduction
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Enable 4-bit precision
    bnb_4bit_quant_type='nf4', # NormalFloat4: optimal for neural network weights
    bnb_4bit_compute_dtype=torch.float16, # Use FP16 for forward and backward passes
    bnb_4bit_use_double_quant=True, # further quantize quantization constants
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, # apply 4-bit quantization
    device_map='auto', # Auto distribute across available gpus
    trust_remote_code=True,
    torch_dtype=torch.float16, #use FP16 for non-quantized operations
    )

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded successfully!")
print(f"Model Parameters: ~{sum(p.numel() for p in model.parameters()) / 1e9:.1f}G")
print(f"Quantized parameters: ~{sum(p.numel() for p in model.parameters() if hasattr(p, 'quant_type')) / 1e9:.1f}G")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded successfully!
Model Parameters: ~1.7G
Quantized parameters: ~1.4G


In [7]:
print(f"Model loaded successfully!")
print(f"Model Parameters: ~{sum(p.numel() for p in model.parameters()) / 1e9:.1f}G")
print(f"Quantized parameters: ~{sum(p.numel() for p in model.parameters() if hasattr(p, 'quant_type')) / 1e9:.1f}G")

Model loaded successfully!
Model Parameters: ~1.7G
Quantized parameters: ~1.4G


#### LoRA Configuration

In [11]:
lora_config = LoraConfig(
    r=16, # Rank: adaptation capacity (16 is reasonable for a reasoning task)
    lora_alpha=32, # scaling factor, typically 2x of rank
    target_modules=['q_proj', 'v_proj'], # focus on attention query/value for reasoning
    lora_dropout=0.1, # Regularization to prevent overfitting
    bias='none', # Skip bias adaptation for simplicity
    task_type=TaskType.CAUSAL_LM, # Causal language modeling task
)

# Apply LoRA configuration to create trainable adapter
model = get_peft_model(model, lora_config)

print(f"LoRA Training Parameters Summary:")
model.print_trainable_parameters()

LoRA Training Parameters Summary:
trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193


/home/monster/Desktop/codes/ML-Workbook/myvenv/lib/python3.12/site-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(


#### Prompt Formatting

In [12]:
# Define structured output format for mathematical reasoning
reasoning_start = '<start_working_out>'
reasoning_end = '<end_working_out>'
solution_start = '<SOLUTION>'
solution_end = '</SOLUTION>'

system_prompt = f""" You are a mathematical reasoining assistant.
When given a math problem:
1. Show your step-by-step work between {reasoning_start} and {reasoning_end}
2. Provide your final numerical answer between {solution_start} and {solution_end}
3. Be precise and show all calculation steps clearly."""


In [13]:
def extract_hash_answer(text):
    """Extract numerical answer from GSM8K format (#### marker)"""
    if "####" not in text:
        return None
    # GSM8K uses format: "Explanation... #### 42"
    return text.split("####")[1].strip()


def process_dataset_example(example):
    """Convert GSM8K example to conversation format for GRPO training"""
    question = example['question']
    answer = extract_hash_answer(example['answer'])

    # create conversation with system prompt for structured reasoning
    prompt = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': question},
    ]

    return {
        'prompt': prompt, # Input conversation
        'answer': answer, # Ground truth for reward function
    }

#### Load dataset

In [14]:
dataset = load_dataset('openai/gsm8k', 'main', split='train')

# Apply conversation formatting to all examples
dataset = dataset.map(process_dataset_example)

print(f"Dataset loaded and processed!")
print(f"Training examples: {len(dataset):,}")
print(f"Sample question: {dataset[0]['prompt'][1]['content']}...")
print(f"Sample answer: {dataset[0]['answer']}")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Dataset loaded and processed!
Training examples: 7,473
Sample question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?...
Sample answer: 72


In [15]:
# show structure of first formatted example 
print(f"Example structure:")
print(f"    Prompt: {len(dataset[0]['prompt'])} messages (system + user)")
print(f"    Answer: {dataset[0]['answer']} (ground truth for rewards)")

Example structure:
    Prompt: 2 messages (system + user)
    Answer: 72 (ground truth for rewards)


#### Multi Reward System Design

In [17]:
match_format = re.compile(
    rf"^[\s]{{0,}}" # optional whitespace at the start
    rf"{reasoning_start}.+?{reasoning_end}.*?" # reasoning section (non-greedy)
    rf"{solution_start}(.+?){solution_end}" # solution section with capture group
    rf"[\s]{{0,}}$", # optional whitespace at the end
    flags=re.MULTILINE | re.DOTALL, # Multi-line matching with . matching newlines
)

match_numbers = re.compile(
    rf"{solution_start}.*?([\d\.]{{1,}})", # Extract numbers from solution section
    flags=re.MULTILINE | re.DOTALL, # Flexible pattern matching
)

# Reward Function 1: Extract Format Compliance
def match_format_exactly(completions, **kwards):
    """
    High reward (3.0) for perfect format adherence
    Ensures model learns the complete structured output pattern
    """
    scores = []
    for completion in completions:
        response = completion[0]['content']
        # check if response matches complete format pattern
        score = 3.0 if match_format.search(response) is not None else 0.0
        scores.append(score)
    return scores


# Reward Function 2: Partial Format Credit
def match_format_approximately(completions, **kwards):
    """
    Graduated scoring for format elements
    Encourages learning individual components even if not perfect
    """
    scores = []
    for completion in completions:
        response = completion[0]['content']
        score = 0

        # Award +0.5 for correct token count, -0.5 for wrong count
        score += 0.5 if response.count(reasoning_start) == 1 else -0.5
        score += 0.5 if response.count(reasoning_end) == 1 else -0.5
        score += 0.5 if response.count(solution_start) == 1 else -0.5
        score += 0.5 if response.count(solution_end) == 1 else -0.5

        scores.append(score)
    return scores

# Reward Function 3: Mathematical Accuracy
def check_answer_correctness(prompts, completions, answer, **kwargs):
    """
    Graduated scoring for mathematical accuracy:
    - 3.0: Exact match
    - 1.5: Within 10% (close answer)
    - 0.5: Within 20% (reasonable attempt)
    - -0.5: Wrong answer (penalty for incorrect math)
    """
    responses = [completion[0]['content'] for completion in completions]

    # Extract answers using format pattern
    extracted_responses = [
        guess.group(1) if (guess := match_format.search(r)) is not None else None for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None: # No extractable answer
            scores.append(0)
            continue

        # Exact string match gets full points
        if guess.strip() == true_answer.strip():
            scores.append(3.0)
        else:
            # Try numerical comparison for partial credit
            try:
                ratio = float(guess) / float(true_answer)
                if 0.9 <= ratio <= 1.1: # Within 10%
                    scores.append(1.5)
                elif 0.8 <= ratio <= 1.2: # within 20%
                    scores.append(0.5)
                else: # wrong answer
                    scores.append(-0.5)
            except (ValueError, ZeroDivisionError):
                scores.append(-0.5) # Invalid numerical format
    return scores


# Reward Function 4: Number Extraction Ability
def check_numbers_extraction(prompts, completions, answer, **kwargs):
    """
    Tests the model's ability to extract numerical values from solutions sections
    Complementary to exact format matching - focuses on parsing capability
    """
    responses = [completion[0]['content'] for completion in completions]

    # Extract numbers from solution sections using number pattern
    extracted_responses = [
        guess.group(1) if (guess := match_numbers.search(r)) is not None else None for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None: # No extractable number
            scores.append(0)
            continue
        
        try:
            # Simple numerical equality check
            true_val = float(true_answer.strip())
            guess_val = float(guess.strip())
            # Binary scoring: correct (1.5) or incorrect (0)
            scores.append(1.5 if guess_val == true_val else 0.0)
        except (ValueError, TypeError):
            scores.append(0)

    return scores

#### GRPO Trianing Configuration

In [20]:
import trl

print(trl.__version__)

1.14.2


In [31]:
# GRPO Configuration
training_args = GRPOConfig(
    learning_rate=5e-6,
    per_device_train_batch_size=2, # small batch for GPU memory constraint
    gradient_accumulation_steps=8,
    # max_prompt_length=1024,
    max_completion_length=1024,
    max_steps=1000,
    logging_steps=1,
    output_dir='./trl_grpo_math_qwen',
    max_grad_norm=0.1,
    report_to='trackio'
)

In [39]:
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[
        match_format_exactly,
        match_format_approximately,
        check_answer_correctness,
        check_numbers_extraction,
    ],
    args=training_args,
    train_dataset=dataset
)

# trainer.train()

print("Training completed successfully")
print(f"Model saved to: {training_args.output_dir}")

Training completed successfully
Model saved to: ./trl_grpo_math_qwen


#### Model Evaluation

In [41]:
def test_model(question, max_length=512):
    """
    Test the trained model on mathematical questions

    Args:
        question (str): Mathematical problem to solve
        max_length: Maximum tokens to generate
    Returns:
        str: Model's structured response with reasoning and solution
    """
    # Format input using conversation template
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': question}
    ]

    # Apply chat template and tokenize
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True, # Add assistant prompt
        tokenize=False # Return strings, not tokens
    )

    # Tokenize and move to appropriate device
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    print(f"Processing: {question}")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.1,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
            length_penalty=1.0,
            early_stopping=True,
        )
    # Decode and extract only the generated portion
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = response[len(text) :].strip()

    return generated_text


sample_question = 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'
expected = "72"

response = test_model(sample_question, max_length=768)

print(f'response: {response}')

Processing: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
response: old in May.

<start_working_out>
- In April, Natalia sold 48 clips.
- In May, she sold half as many clips as in April.
- Therefore, the number of clips sold in May = \(\frac{48}{2} = 24\).
</start_working_out>

Now we'll calculate the total number of clips sold in both months combined.

<end_working_out>
- Total clips sold = Clips sold in April + Clips sold in May
- Total clips sold = 48 + 24
- Total clips sold = 72

<SOLUTION>72</SOLUTION>

</SOLUTION>


In [ ]:
has_reasoning = reasoning_start in response and reasoning_end in response
has_solution = solution_start in response and solution_end in response

print(f"\nFormat Check:")
print(f"Reasoning section: {has_reasoning}")
print(f"Solution section: {has_solution}")

if has_solution:
    try:
        solution_text = response.split(solution_start)[1].split(solution_end)[0].strip()
        extracted_number = ''.join(filter(str.isdigit, solution_text))
        expected_number = ''.join(filter(str.isdigit, expected))
        is_correct = extracted_number == expected_number

        print(f"Extracted: {solution_text}")
        print(f"Expected: {expected}")
        print(f"Correct: {is_correct}")
    except:
        print('Could not extract solution')


Format Check:
Reasoning section: True
Solution section: True
Extracted: 72
Expected: 72
Correct: True


: 